# Common agent evaluation for Team A

Group 9

Each team runs this notebook separately in a fresh Colab CPU session using its own submission archive. Upload the official competition ZIP and your own `.tar.gz` agent package. No repository connection, Kaggle credentials or package installation is required.

The team name is preset and the agent label comes from the uploaded archive filename. No code changes are needed. The benchmark uses the same official sample deck, three simple reference policies and 40 matches per opponent. Each match has at most three games and stops when either player wins twice. This produces 120 matches and between 240 and 360 games when every game completes without a draw. Draws can increase the number of games to 360.

The exported comparison summary contains scores and file fingerprints. It does not contain your agent, model weights or observations. Keep your private input files and detailed logs within your Kaggle team. Share only evidence permitted by the competition rules and your course instructions.

This is a local SDK benchmark. It does not submit to Kaggle or reproduce its hosted rating system. The notebook has not been run with the native engine here.


## 1. Labels and fixed settings

This copy is preset for Team A (`GROUP_PROGRAMMING_09_a`). Open a fresh Colab CPU session and run all cells. Upload the official competition ZIP and your own agent submission archive when prompted. Keep the evaluation settings unchanged.


In [1]:
TEAM_NAME = 'GROUP_PROGRAMMING_09_a'
AGENT_LABEL = None

import ast
import csv
import importlib.metadata
import json
import os
import platform
import shutil
import signal
import subprocess
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

assert TEAM_NAME in ('GROUP_PROGRAMMING_09_a', 'GROUP_PROGRAMMING_09_b')
assert platform.system() == 'Linux' and platform.machine() == 'x86_64', 'Use a fresh Colab CPU session.'
os.environ.update(CUDA_VISIBLE_DEVICES='-1', OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
work_dir = Path('/content/ptcg_common_evaluation') / RUN_ID
output_dir = work_dir / 'private_results'
source_dir = output_dir / 'source'
sdk_dir = work_dir / 'official_sample'
candidate_dir = work_dir / 'own_agent'
for directory in (source_dir, sdk_dir):
    directory.mkdir(parents=True, exist_ok=False)
print('Run:', RUN_ID)
print('Team:', TEAM_NAME)
print('Runtime:', sys.version.split()[0], platform.machine(), '| CPU')


Run: 20261126T223506Z_daef86ed
Team: GROUP_PROGRAMMING_09_a
Runtime: 3.13.18 x86_64 | CPU


## 2. Benchmark code

The runner checks every returned action. Invalid selections, missing files, engine errors and incomplete processes remain visible in the results. It does not repair actions or substitute another policy.


In [2]:
SOURCES = {'common_benchmark_core.py': '"""Neutral shared benchmark helpers. No team model or private policy is included."""\nfrom contextlib import contextmanager\nimport hashlib\nimport inspect\nimport json\nimport math\nimport os\nfrom pathlib import Path, PurePosixPath\nimport random\nimport sys\nimport tarfile\n\nOPPONENTS = (\'sample_random\', \'first_options\', \'last_options\')\nDRAW_RULE = \'At most three games; stop at two wins. At the cap, more game wins takes the match; equal wins is a draw.\'\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef digest_bytes(value):\n    return hashlib.sha256(value).hexdigest()\n\n\ndef digest_json(value):\n    return digest_bytes(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode())\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for block in iter(lambda: handle.read(1024*1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef deck_ids(path):\n    values = [int(line) for line in Path(path).read_text(encoding=\'utf-8-sig\').splitlines() if line.strip()]\n    if len(values) != 60:\n        raise ValueError(\'deck.csv must contain 60 integer card IDs.\')\n    return values\n\n\ndef extract_agent(path, destination, maximum_bytes=512*1024*1024, maximum_files=5000):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=False)\n    seen, total = set(), 0\n    with tarfile.open(path, \'r:gz\') as archive:\n        for member in archive:\n            relative = PurePosixPath(member.name)\n            if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\' in member.name:\n                raise ValueError(\'Unsafe archive path: \' + member.name)\n            if member.isdir():\n                continue\n            if not member.isfile() or member.issym() or member.islnk():\n                raise ValueError(\'Only regular files and directories are supported.\')\n            name = relative.as_posix()\n            if name in seen or name == \'.\':\n                raise ValueError(\'Duplicate or invalid archive file: \' + name)\n            seen.add(name)\n            total += member.size\n            if len(seen) > maximum_files or total > maximum_bytes:\n                raise ValueError(\'Archive exceeds the notebook extraction budget.\')\n            target = destination.joinpath(*relative.parts)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            content = archive.extractfile(member).read()\n            if len(content) != member.size:\n                raise ValueError(\'Archive member size mismatch.\')\n            target.write_bytes(content)\n            target.chmod(0o755 if member.mode & 0o111 else 0o644)\n    if not {\'main.py\', \'deck.csv\'} <= seen:\n        raise ValueError(\'The archive must contain main.py and deck.csv at its root.\')\n    return {\'unpacked_bytes\': total, \'files\': {name: file_hash(destination / name) for name in sorted(seen)}}\n\n\ndef validate_action(action, selection):\n    if not isinstance(action, list) or not all(type(index) is int for index in action):\n        raise ValueError(\'Actions must be a list of Python integer option indices.\')\n    options = selection[\'option\']\n    if not selection[\'minCount\'] <= len(action) <= selection[\'maxCount\']:\n        raise ValueError(\'Invalid selection count.\')\n    if len(set(action)) != len(action) or any(index < 0 or index >= len(options) for index in action):\n        raise ValueError(\'Duplicate or out-of-range option indices.\')\n\n\ndef reference_action(name, observation, deck, rng):\n    if observation.get(\'select\') is None:\n        return list(deck)\n    selection = observation[\'select\']\n    count, total = selection[\'maxCount\'], len(selection[\'option\'])\n    if name == \'sample_random\':\n        action = rng.sample(range(total), count)\n    elif name == \'first_options\':\n        action = list(range(count))\n    elif name == \'last_options\':\n        action = list(range(total-count, total))\n    else:\n        raise ValueError(\'Unknown reference opponent.\')\n    validate_action(action, selection)\n    return action\n\n\n@contextmanager\ndef package_context(directory):\n    previous_cwd, previous_path = os.getcwd(), list(sys.path)\n    os.chdir(directory)\n    sys.path.insert(0, str(directory))\n    try:\n        yield\n    finally:\n        os.chdir(previous_cwd)\n        sys.path[:] = previous_path\n\n\ndef load_candidate(directory):\n    directory = Path(directory).resolve()\n    path = directory / \'main.py\'\n    namespace = {}\n    with package_context(directory):\n        exec(compile(path.read_text(encoding=\'utf-8-sig\'), str(path), \'exec\'), namespace)\n    callables = [value for value in namespace.values() if callable(value)]\n    function = namespace.get(\'agent\')\n    if not callable(function) or not callables or callables[-1] is not function:\n        raise ValueError(\'agent must be the final callable in main.py.\')\n    signature = inspect.signature(function)\n    configuration = {\'__raw_path__\': str(path)}\n    sentinel = object()\n    try:\n        signature.bind(sentinel, configuration)\n        mode = \'two_positional_arguments\'\n    except TypeError:\n        try:\n            signature.bind(sentinel, configuration=configuration)\n            mode = \'configuration_keyword\'\n        except TypeError:\n            signature.bind(sentinel)\n            mode = \'one_argument\'\n\n    def invoke(observation, seed):\n        configuration[\'seed\'] = seed\n        with package_context(directory):\n            if mode == \'two_positional_arguments\':\n                return function(observation, configuration)\n            if mode == \'configuration_keyword\':\n                return function(observation, configuration=configuration)\n            return function(observation)\n\n    return invoke, mode\n\n\ndef match_outcome(outcomes):\n    if not outcomes or any(value not in (\'win\', \'loss\', \'draw\') for value in outcomes):\n        return \'error\'\n    wins, losses = outcomes.count(\'win\'), outcomes.count(\'loss\')\n    if wins >= 2:\n        return \'win\'\n    if losses >= 2:\n        return \'loss\'\n    if len(outcomes) < 3:\n        return None\n    return \'win\' if wins > losses else \'loss\' if losses > wins else \'draw\'\n\n\ndef wilson(wins, total):\n    if not total:\n        return None\n    z = 1.959963984540054\n    p, denominator = wins/total, 1+z*z/total\n    center = (p+z*z/(2*total))/denominator\n    half = z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/denominator\n    return [max(0.0, center-half), min(1.0, center+half)]\n\n\ndef score_bound(mean, total):\n    if not total or mean is None:\n        return None\n    radius = math.sqrt(math.log(40)/(2*total))\n    return [max(0.0, mean-radius), min(1.0, mean+radius)]\n\n\ndef summarize(report, requested, process_ok):\n    matches = report.get(\'matches\', [])\n    valid = [row for row in matches if row.get(\'outcome\') in (\'win\', \'loss\', \'draw\')]\n    wins = sum(row[\'outcome\'] == \'win\' for row in valid)\n    losses = sum(row[\'outcome\'] == \'loss\' for row in valid)\n    draws = sum(row[\'outcome\'] == \'draw\' for row in valid)\n    errors = sum(row.get(\'outcome\') == \'error\' for row in matches)\n    games = [game for row in matches for game in row.get(\'games\', [])]\n    invalid = sum(game.get(\'invalid_responses\', 0) for game in games)\n    rejected = sum(game.get(\'engine_rejections\', 0) for game in games)\n    complete = (process_ok and report.get(\'contract_passed\') is True and report.get(\'status\') == \'passed\'\n                and len(valid) == requested and errors == 0 and invalid == rejected == 0)\n    score = (wins+.5*draws)/len(valid) if valid else None\n    decisions = sum(game.get(\'candidate_decisions\', 0) for game in games)\n    return {\'requested_matches\': requested, \'valid_matches\': len(valid), \'error_matches\': errors,\n            \'unfinished_or_unstarted_matches\': requested-len(valid)-errors,\n            \'wins\': wins, \'losses\': losses, \'draws\': draws,\n            \'win_rate\': wins/len(valid) if valid else None, \'score_rate\': score,\n            \'win_rate_wilson_95\': wilson(wins, len(valid)),\n            \'score_rate_hoeffding_95\': score_bound(score, len(valid)),\n            \'games_played\': len(games), \'invalid_responses\': invalid, \'engine_rejections\': rejected,\n            \'candidate_decisions\': decisions,\n            \'candidate_mean_decision_ms\': sum(game.get(\'candidate_total_decision_ms\', 0) for game in games)/decisions if decisions else None,\n            \'candidate_max_decision_ms\': max([game.get(\'candidate_max_decision_ms\', 0) for game in games] or [0]),\n            \'complete_and_valid\': complete}\n', 'common_benchmark_worker.py': '"""Run one public reference comparison in an isolated process."""\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport traceback\n\nfrom common_benchmark_core import (deck_ids, load_candidate, match_outcome,\n                                   reference_action, save_json, validate_action)\n\n\ndef run_game(candidate, opponent, deck, seed, seat, reverse, max_steps, api):\n    start, select, finish, Battle = api\n    random.seed(seed)\n    rng = random.Random(seed)\n    began = time.perf_counter()\n    row = dict(python_seed=seed, candidate_seat=seat, reverse_player=reverse,\n               status=\'running\', outcome=\'error\', result_code=None, steps=0,\n               invalid_responses=0, engine_rejections=0, candidate_decisions=0,\n               candidate_total_decision_ms=0.0, candidate_max_decision_ms=0.0,\n               error=\'\', traceback=\'\', cleanup_error=\'\')\n\n    def candidate_call(observation):\n        before = time.perf_counter()\n        try:\n            return candidate(observation, seed)\n        finally:\n            elapsed = (time.perf_counter() - before)*1000\n            row[\'candidate_decisions\'] += 1\n            row[\'candidate_total_decision_ms\'] += elapsed\n            row[\'candidate_max_decision_ms\'] = max(row[\'candidate_max_decision_ms\'], elapsed)\n\n    try:\n        initial = {\'select\': None, \'current\': None, \'logs\': [], \'search_begin_input\': None}\n        returned = candidate_call(dict(initial))\n        if not isinstance(returned, list) or any(type(card) is not int for card in returned) or returned != deck:\n            row[\'invalid_responses\'] += 1\n            raise ValueError(\'Candidate must return the fixed 60-card sample deck in the recorded order.\')\n        observation, start_info = start(list(deck), list(deck), reverse)\n        if observation is None:\n            raise RuntimeError(\'Battle start failed: \' + str(start_info))\n        for step in range(max_steps+1):\n            current = observation.get(\'current\')\n            if current is None:\n                raise ValueError(\'Simulator returned no current state.\')\n            result = current[\'result\']\n            if result != -1:\n                if result not in (0, 1, 2):\n                    raise ValueError(\'Unknown terminal result: \' + str(result))\n                row.update(status=\'passed\', result_code=result,\n                           outcome=\'draw\' if result == 2 else \'win\' if result == seat else \'loss\')\n                break\n            if step == max_steps:\n                raise RuntimeError(\'Selection limit reached before the game ended.\')\n            player, selection = current[\'yourIndex\'], observation.get(\'select\')\n            if player not in (0, 1) or selection is None:\n                raise ValueError(\'Unexpected acting player or missing selection.\')\n            action = (candidate_call(observation) if player == seat else\n                      reference_action(opponent, observation, deck, rng))\n            try:\n                validate_action(action, selection)\n            except Exception:\n                row[\'invalid_responses\'] += 1\n                raise\n            try:\n                observation = select(action)\n                if observation is None:\n                    raise RuntimeError(\'Simulator rejected the selection.\')\n            except Exception:\n                row[\'engine_rejections\'] += 1\n                raise\n            row[\'steps\'] += 1\n    except Exception as error:\n        row.update(status=\'error\', outcome=\'error\', error=type(error).__name__ + \': \' + str(error),\n                   traceback=traceback.format_exc())\n    finally:\n        try:\n            if Battle.battle_ptr:\n                finish()\n        except Exception as error:\n            row.update(status=\'cleanup_error\', outcome=\'error\', cleanup_error=str(error))\n        finally:\n            Battle.battle_ptr = None\n        row[\'seconds\'] = time.perf_counter() - began\n    return row\n\n\ndef worker(job_path):\n    job = json.loads(Path(job_path).read_text())\n    output = Path(job[\'output_dir\'])\n    report = {\'status\': \'running\', \'contract_passed\': False, \'matches\': [], \'error\': None}\n\n    def checkpoint():\n        save_json(output / \'report.json\', report)\n        fields = [\'match_id\', \'game_id\', \'outcome\', \'status\', \'python_seed\', \'candidate_seat\',\n                  \'reverse_player\', \'result_code\', \'steps\', \'seconds\', \'invalid_responses\',\n                  \'engine_rejections\', \'candidate_decisions\', \'candidate_total_decision_ms\',\n                  \'candidate_max_decision_ms\', \'error\', \'cleanup_error\']\n        temporary = output / \'games.csv.tmp\'\n        with temporary.open(\'w\', newline=\'\', encoding=\'utf-8\') as handle:\n            writer = csv.DictWriter(handle, fieldnames=fields, extrasaction=\'ignore\', lineterminator=\'\\n\')\n            writer.writeheader()\n            for match in report[\'matches\']:\n                writer.writerows(match[\'games\'])\n        temporary.replace(output / \'games.csv\')\n\n    checkpoint()\n    try:\n        os.environ[\'PTCG_CG_DIR\'] = str(Path(job[\'sdk_dir\']) / \'cg\')\n        sys.path.insert(0, job[\'sdk_dir\'])\n        from cg.game import battle_start, battle_select, battle_finish\n        from cg.sim import Battle\n        api = battle_start, battle_select, battle_finish, Battle\n        random.seed(job[\'seed_base\'])\n        candidate, mode = load_candidate(job[\'candidate_dir\'])\n        deck = deck_ids(Path(job[\'sdk_dir\']) / \'deck.csv\')\n        if deck_ids(Path(job[\'candidate_dir\']) / \'deck.csv\') != deck:\n            raise ValueError(\'Candidate deck.csv differs from the common sample deck.\')\n        initial = {\'select\': None, \'current\': None, \'logs\': [], \'search_begin_input\': None}\n        returned = candidate(initial, job[\'seed_base\'])\n        if not isinstance(returned, list) or any(type(card) is not int for card in returned) or returned != deck:\n            raise ValueError(\'Candidate deck callback failed the fixed deck check.\')\n        report.update(contract_passed=True, callback_mode=mode)\n        checkpoint()\n        for match_index in range(job[\'matches_per_opponent\']):\n            match = {\'match_id\': match_index+1, \'outcome\': None, \'games\': []}\n            report[\'matches\'].append(match)\n            checkpoint()\n            for game_index in range(3):\n                seed = job[\'seed_base\'] + match_index*3 + game_index\n                seat = (match_index + game_index) % 2\n                reverse = bool((match_index//2 + game_index//2) % 2)\n                game = run_game(candidate, job[\'opponent\'], deck, seed, seat, reverse, job[\'max_steps\'], api)\n                game.update(match_id=match_index+1, game_id=game_index+1)\n                match[\'games\'].append(game)\n                match[\'outcome\'] = match_outcome([row[\'outcome\'] for row in match[\'games\']])\n                checkpoint()\n                if match[\'outcome\'] is not None:\n                    break\n            print(\'Match\', match_index+1, \'/\', job[\'matches_per_opponent\'], match[\'outcome\'], flush=True)\n        report[\'status\'] = (\'passed\' if all(row[\'outcome\'] in (\'win\', \'loss\', \'draw\')\n                                           for row in report[\'matches\']) else \'failed\')\n    except Exception as error:\n        report.update(status=\'error\', error={\'type\': type(error).__name__, \'message\': str(error),\n                                            \'traceback\': traceback.format_exc()})\n    finally:\n        checkpoint()\n    return 0 if report[\'status\'] == \'passed\' else 1\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(worker(sys.argv[1]))\n'}
for name, source in SOURCES.items():
    ast.parse(source, filename=name)
    (source_dir / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(source_dir))
from common_benchmark_core import (OPPONENTS, DRAW_RULE, deck_ids, digest_json,
                                   extract_agent, file_hash, save_json, summarize)

OFFICIAL_ZIP_SHA256 = '6fe557e77ca21b86dea9056d7f4531cbeb328abc1b25aecf9c3a4e2b7d5b2c87'
CANONICAL_DECK_SHA256 = 'c3d72ecd696951d73dbb99a1b575e2bd185b43b0f2da02f3e25be664bc68d34c'
PROTOCOL = {
    'name': 'group9_common_bo3_v1',
    'unit': 'externally_controlled_match',
    'matches_per_opponent': 40,
    'opponents': list(OPPONENTS),
    'reference_policy': 'Choose maxCount indices uniformly without replacement, in prefix order, or in suffix order.',
    'deck_policy': 'Both players return the official 60 sample card IDs in the same order.',
    'canonical_deck_sha256': CANONICAL_DECK_SHA256,
    'official_zip_sha256': OFFICIAL_ZIP_SHA256,
    'match_rule': DRAW_RULE,
    'match_score': 'Win=1, draw=0.5, loss=0. Aggregate is the unweighted mean of the three opponent score rates.',
    'seed_base': 91000,
    'seed_schedule': 'seed_base + 3*match_index + game_index; zero based; identical for each opponent.',
    'candidate_seat': '(match_index + game_index) % 2',
    'reverse_player': 'bool((match_index//2 + game_index//2) % 2)',
    'maximum_selections_per_game': 10000,
    'process_timeout_seconds_per_opponent': 1800,
    'process_policy': 'One fresh Python process per reference; agent module persists within that process across matches.',
    'callback_policy': 'No __file__ in entry namespace; package working directory and sys.path; one or two callback arguments.',
    'randomness': 'Seed Python random at each game; reference uses a separate random.Random. Native and framework RNG control is unverified.',
    'hardware': 'Linux x86_64 CPU; CUDA hidden; numerical library thread limits set to one.',
    'archive_budget': {'maximum_unpacked_bytes': 536870912, 'maximum_files': 5000},
    'eligibility': 'All 120 matches complete, no invalid responses or engine errors, and all three worker processes succeed.',
    'source_sha256': {name: file_hash(source_dir / name) for name in SOURCES},
}
PROTOCOL_ID = digest_json(PROTOCOL)
save_json(output_dir / 'protocol.json', {'protocol_id': PROTOCOL_ID, 'settings': PROTOCOL})
print('Protocol:', PROTOCOL_ID)
print('Matches:', len(OPPONENTS) * PROTOCOL['matches_per_opponent'])


Protocol: b5071f213a05c0f8ee5327291a90c82f7b7099d637cce01fb5452758cfaf2f8e
Matches: 120


## 3. Upload your two input files

Upload the same official competition ZIP used for the earlier notebooks. Then upload only your own submission `.tar.gz`, with `main.py`, `deck.csv` and its required assets at the archive root. Use a package you trust. Its Python code will execute in your Colab session.

The deck check compares parsed card IDs, so different CSV line endings are accepted. A different deck or simulator version stops the run. An agent that needs an unavailable dependency must be fixed within its own team before this common benchmark.


In [3]:
from google.colab import files

def upload_one(description, predicate):
    print(description)
    uploaded = files.upload()
    paths = [Path.cwd() / name for name in uploaded]
    matching = [path.resolve() for path in paths if path.is_file() and predicate(path)]
    if len(matching) != 1:
        raise ValueError('Upload exactly one matching file for this prompt.')
    return matching[0]

official_zip = upload_one('Upload the official competition ZIP.',
                          lambda path: path.suffix == '.zip' and file_hash(path) == OFFICIAL_ZIP_SHA256)
agent_archive = upload_one('Upload your own agent submission .tar.gz.',
                          lambda path: path.name.endswith('.tar.gz'))
AGENT_LABEL = agent_archive.name[:-len('.tar.gz')]
assert AGENT_LABEL.strip(), 'Use a descriptive agent archive filename.'
print('Agent:', AGENT_LABEL)
prefix = 'sample_submission/sample_submission/sample_submission/'
with zipfile.ZipFile(official_zip) as archive:
    selected = [name for name in archive.namelist() if name.startswith(prefix) and not name.endswith('/')]
    if not selected:
        raise ValueError('Official sample package not found in this ZIP.')
    for name in selected:
        relative = PurePosixPath(name[len(prefix):])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Unexpected SDK archive path.')
        target = sdk_dir.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
deck = deck_ids(sdk_dir / 'deck.csv')
assert digest_json(deck) == CANONICAL_DECK_SHA256
package_info = extract_agent(agent_archive, candidate_dir, **{
    'maximum_bytes': PROTOCOL['archive_budget']['maximum_unpacked_bytes'],
    'maximum_files': PROTOCOL['archive_budget']['maximum_files']})
assert deck_ids(candidate_dir / 'deck.csv') == deck, 'Use the fixed sample deck for this comparison.'
versions = {}
for package in ('numpy', 'torch', 'scipy', 'scikit-learn', 'matplotlib'):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None
runtime = {'python': platform.python_version(), 'platform': platform.platform(),
           'machine': platform.machine(), 'packages': versions}
RUNTIME_ID = digest_json(runtime)
candidate_sha256 = file_hash(agent_archive)
run_info = {'run_id': RUN_ID, 'team': TEAM_NAME, 'agent': AGENT_LABEL,
            'protocol_id': PROTOCOL_ID, 'runtime_id': RUNTIME_ID, 'runtime': runtime,
            'candidate_archive_sha256': candidate_sha256, 'candidate_package': package_info,
            'candidate_deck_raw_sha256': file_hash(candidate_dir / 'deck.csv'),
            'canonical_deck_sha256': digest_json(deck),
            'sdk_files': {str(path.relative_to(sdk_dir)): file_hash(path)
                          for path in sorted(sdk_dir.rglob('*')) if path.is_file()}}
save_json(output_dir / 'run_info.json', run_info)
print('Inputs verified. Candidate archive:', candidate_sha256)
print('Runtime fingerprint:', RUNTIME_ID)


Upload the official competition ZIP.


Saving the-pokemon-company-ptcg-ai-battle-challenge-playground.zip to the-pokemon-company-ptcg-ai-battle-challenge-playground.zip
Upload your own agent submission .tar.gz.


Saving A-R2.tar.gz to A-R2.tar.gz
Agent: A-R2
Inputs verified. Candidate archive: c420603d62a97462f135f1b5579ed91dcfb573138960bdad138dd5af3b108c64
Runtime fingerprint: 9f51ad198a2a293ea5615df6a38654767c52eafbba358f8a671277eada6ffb73


## 4. Run the common comparison

Each reference gets 40 matches. The reference policies are deliberately simple controls; they cannot establish that an agent is best against strong opponents. A match stops at two game wins or after three games. At the cap, the side with more wins takes the match; equal wins produce a draw. Game draws consume a slot.

The runner alternates seats and the SDK reverse flag. It starts a new SDK battle for every game and does not inject previous match results into observations. The simulator round field is unchanged. These results must be compared with other runs of this notebook, rather than the earlier individual game results.

Python seeds and schedules match between teams. Native engine randomness is not controlled, so these are independent benchmark samples, not identical shuffled deals. The engine runs locally in Colab and is not the hosted Kaggle harness.

Progress appears as matches finish. Each opponent has a 30 minute process limit. The combined limit is 90 minutes; this is a cutoff, not an estimated completion time. Timing is diagnostic because Colab CPUs differ. If you interrupt the run, continue to the export cells to save the partial results.


In [4]:
process_records = {}
interrupted = False
for opponent in OPPONENTS:
    phase_dir = output_dir / opponent
    phase_dir.mkdir(parents=True, exist_ok=True)
    job = {'sdk_dir': str(sdk_dir), 'candidate_dir': str(candidate_dir),
           'output_dir': str(phase_dir), 'opponent': opponent,
           'matches_per_opponent': PROTOCOL['matches_per_opponent'],
           'seed_base': PROTOCOL['seed_base'], 'max_steps': PROTOCOL['maximum_selections_per_game']}
    save_json(phase_dir / 'job.json', job)
    process_info = {'returncode': None, 'timed_out': False, 'interrupted': False, 'error': None}
    environment = os.environ.copy()
    environment.update(PYTHONUNBUFFERED='1', PYTHONHASHSEED='0', CUDA_VISIBLE_DEVICES='-1',
                       OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
    process = None
    print('Starting:', opponent, flush=True)
    try:
        with (phase_dir / 'stdout.txt').open('w') as stdout, (phase_dir / 'stderr.txt').open('w') as stderr:
            process = subprocess.Popen([sys.executable, str(source_dir / 'common_benchmark_worker.py'),
                                        str(phase_dir / 'job.json')], cwd=work_dir, env=environment,
                                       stdout=stdout, stderr=stderr, start_new_session=True)
            started, offset, last_update = time.monotonic(), 0, time.monotonic()
            while process.poll() is None:
                if time.monotonic()-started > PROTOCOL['process_timeout_seconds_per_opponent']:
                    process_info['timed_out'] = True
                    break
                with (phase_dir / 'stdout.txt').open() as progress:
                    progress.seek(offset)
                    new_text = progress.read()
                    offset = progress.tell()
                if new_text:
                    print(new_text, end='', flush=True)
                    last_update = time.monotonic()
                elif time.monotonic()-last_update >= 30:
                    print('Still running:', opponent, '| elapsed seconds:', round(time.monotonic()-started), flush=True)
                    last_update = time.monotonic()
                time.sleep(2)
    except KeyboardInterrupt:
        process_info['interrupted'] = True
        interrupted = True
    except Exception as error:
        process_info['error'] = type(error).__name__ + ': ' + str(error)
    finally:
        if process is not None:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            process_info['returncode'] = process.wait()
        process_records[opponent] = process_info
        save_json(phase_dir / 'process.json', process_info)
    print('Process result:', opponent, process_info, flush=True)
    if interrupted:
        print('Interrupted. Continue to the summary and export cells.')
        break


Starting: sample_random
Match 1 / 40 win
Match 2 / 40 win
Match 3 / 40 win
Match 4 / 40 win
Match 5 / 40 win
Match 6 / 40 win
Match 7 / 40 win
Match 8 / 40 win
Match 9 / 40 win
Match 10 / 40 win
Match 11 / 40 win
Match 12 / 40 win
Match 13 / 40 win
Match 14 / 40 win
Match 15 / 40 win
Match 16 / 40 win
Match 17 / 40 win
Match 18 / 40 win
Match 19 / 40 win
Match 20 / 40 win
Process result: sample_random {'returncode': 0, 'timed_out': False, 'interrupted': False, 'error': None}
Starting: first_options
Match 1 / 40 win
Match 2 / 40 loss
Match 3 / 40 loss
Match 4 / 40 loss
Match 5 / 40 loss
Match 6 / 40 win
Match 7 / 40 win
Match 8 / 40 loss
Match 9 / 40 win
Match 10 / 40 loss
Match 11 / 40 loss
Match 12 / 40 loss
Match 13 / 40 loss
Match 14 / 40 win
Match 15 / 40 win
Match 16 / 40 loss
Match 17 / 40 win
Match 18 / 40 win
Match 19 / 40 win
Match 20 / 40 loss
Match 21 / 40 win
Match 22 / 40 win
Match 23 / 40 loss
Match 24 / 40 win
Match 25 / 40 win
Process result: first_options {'returncode'

## 5. Read the results

Score rate gives one point for a match win and half a point for a draw. The overall score is the mean of the three opponent score rates. It is left blank unless all planned matches and checks succeed. Partial rates remain available for diagnosis.

Wilson intervals describe binary win rates, counting draws as nonwins. Hoeffding bounds describe match score rates. Both assume independent matches and do not adjust for repeated model selection, persistent agent state or native engine dependence. Use them as uncertainty summaries, not proof that a small score difference identifies a stronger agent.


In [5]:
rows = []
for opponent in OPPONENTS:
    phase_dir = output_dir / opponent
    report_path = phase_dir / 'report.json'
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    process_path = phase_dir / 'process.json'
    process_info = json.loads(process_path.read_text()) if process_path.is_file() else {}
    process_ok = (process_info.get('returncode') == 0 and not process_info.get('timed_out', False)
                  and not process_info.get('interrupted', False) and process_info.get('error') is None)
    result = summarize(report, PROTOCOL['matches_per_opponent'], process_ok)
    row = {'run_id': RUN_ID, 'team': TEAM_NAME, 'agent': AGENT_LABEL,
           'protocol_id': PROTOCOL_ID, 'runtime_id': RUNTIME_ID,
           'candidate_archive_sha256': candidate_sha256, 'opponent': opponent, **result}
    rows.append(row)
    save_json(phase_dir / 'summary.json', row)
    print(opponent, '| wins:', row['wins'], '| losses:', row['losses'], '| draws:', row['draws'],
          '| valid:', row['valid_matches'], '/', row['requested_matches'],
          '| score:', None if row['score_rate'] is None else round(row['score_rate'], 4),
          '| complete:', row['complete_and_valid'])
    if not row['complete_and_valid']:
        print('Review private_results/' + opponent + '/stderr.txt and report.json before rerunning.')
eligible = all(row['complete_and_valid'] for row in rows)
overall = sum(row['score_rate'] for row in rows)/len(rows) if eligible else None
comparison = {'run_id': RUN_ID, 'team': TEAM_NAME, 'agent': AGENT_LABEL,
              'protocol_id': PROTOCOL_ID, 'runtime_id': RUNTIME_ID,
              'candidate_archive_sha256': candidate_sha256,
              'selection_eligible': eligible, 'macro_match_score_rate': overall,
              'opponents': rows,
              'limitations': ['Three simple controls only.', 'Native RNG is not seeded.',
                              'No hosted Kaggle acceptance check.', 'Uncertainty assumes independent matches.',
                              'Compare identical protocol IDs and review runtime differences.']}
print('Selection eligible:', eligible)
print('Mean match score rate:', overall)


sample_random | wins: 40 | losses: 0 | draws: 0 | valid: 40 / 40 | score: 1.0 | complete: True
first_options | wins: 22 | losses: 18 | draws: 0 | valid: 40 / 40 | score: 0.55 | complete: True
last_options | wins: 40 | losses: 0 | draws: 0 | valid: 40 / 40 | score: 1.0 | complete: True
Selection eligible: True
Mean match score rate: 0.85


## 6. Save the evidence

Download the comparison CSV and JSON for the group report. Both teams should use the same protocol fingerprint, confirm complete runs and review runtime differences before comparing scores. If results are close, report the uncertainty and keep the model choice tentative. Freeze candidates before the final common evaluation instead of repeatedly choosing settings from these scores.

The private ZIP contains detailed logs and the neutral evaluation code. It excludes uploaded archives, extracted agent files, model weights and the official SDK. Review logs before sharing because an agent error can expose source lines or private file paths. Download the executed notebook separately through Colab File, Download, Download .ipynb. Keep both Kaggle teams' own submission histories and scores for the final report.


In [6]:
summary_dir = work_dir / 'comparison_summary'
summary_dir.mkdir(exist_ok=True)
save_json(summary_dir / 'comparison_summary.json', comparison)
with (summary_dir / 'comparison_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]), lineterminator='\n')
    writer.writeheader()
    for row in rows:
        writer.writerow({key: json.dumps(value) if isinstance(value, (list, dict)) else value
                         for key, value in row.items()})
save_json(output_dir / 'comparison_summary.json', comparison)
shutil.copy2(summary_dir / 'comparison_summary.csv', output_dir / 'comparison_summary.csv')
save_json(output_dir / 'completion.json', {'selection_eligible': eligible, 'protocol_id': PROTOCOL_ID,
                                         'run_id': RUN_ID, 'macro_match_score_rate': overall})
private_zip = Path(shutil.make_archive(str(work_dir / ('Common_Evaluation_' + RUN_ID + '_private_results')),
                                      'zip', output_dir))
print('Comparison files:', summary_dir)
print('Private results:', private_zip)
print('Save these downloads before ending the Colab session.')
files.download(str(summary_dir / 'comparison_summary.csv'))
files.download(str(summary_dir / 'comparison_summary.json'))
files.download(str(private_zip))


Comparison files: /content/ptcg_common_evaluation/20261126T223506Z_daef86ed/comparison_summary
Private results: /content/ptcg_common_evaluation/20261126T223506Z_daef86ed/Common_Evaluation_20261126T223506Z_daef86ed_private_results.zip
Save these downloads before ending the Colab session.
